# 3 · Gold — agregación con PySpark + Iceberg (`gold_agg.ipynb`)

Respaldo manual de la tarea **`gold_agg`** del DAG. Ejecuta **la misma función** que Airflow (`jobs/gold_agg.py`).

- Cruza las dos tablas históricas de Silver: **`post_hist ⋈ votes_hist`** (por `post_id`).
- Regla de negocio: métricas de votos por post (total, positivos, negativos, favoritos, balance, primer y último voto).
- Escribe **`nessie.gold.vote_stats_per_post`** (Iceberg) con **`MERGE INTO`** y la columna **`fecha_cargue`**.

La misma agregación, escrita en SQL de Trino sobre Silver, está en `consultas/trino_replica_gold.sql`.

In [1]:
NOMBRE_APP = "gold_agg_notebook"
import os, sys
sys.path.insert(0, os.environ.get("JOBS_DIR", "/workspace/jobs"))   # carpeta jobs/ del proyecto
import comun

# Sesión de Spark contra el clúster (spark://spark-master:7077): aparece en la UI de Spark (9090)
spark = comun.crear_spark(NOMBRE_APP, master=os.environ["SPARK_MASTER"])
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/14 15:18:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Agregación y MERGE en `nessie.gold.vote_stats_per_post`

In [2]:
import gold_agg
gold_agg.main(spark)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


nessie.gold.vote_stats_per_post: 1,062,656 filas


+-----------------------+-------------------+---------+---------+--------+
|committed_at           |snapshot_id        |operation|agregadas|borradas|
+-----------------------+-------------------+---------+---------+--------+
|2026-10-14 15:19:06.503|9174103377771718017|overwrite|1062656  |1062656 |
|2026-10-14 15:03:39.507|5723100651381584663|overwrite|1062656  |NULL    |
+-----------------------+-------------------+---------+---------+--------+



+---------+-----------------+------+------+---------+---------+
|anio_post|        post_type| posts| votos|positivos|negativos|
+---------+-----------------+------+------+---------+---------+
|     2020|extracto_wiki_tag|   304|   318|        0|        0|
|     2020|         pregunta|221567|405525|   254971|    91690|
|     2020|        respuesta|348664|714418|   482237|    24728|
|     2020|         wiki_tag|   221|   227|        0|        0|
|     2021|extracto_wiki_tag|   125|   127|        0|        0|
|     2021|         pregunta|185736|318066|   197450|    84593|
|     2021|        respuesta|305940|574607|   362080|    20583|
|     2021|         wiki_tag|    99|   100|        0|        0|
+---------+-----------------+------+------+---------+---------+



+--------+------+-----------+-------------+
| post_id|titulo|total_votos|balance_votos|
+--------+------+-----------+-------------+
|66081492|  NULL|        493|          492|
|65924342|  NULL|        266|          266|
|60071551|  NULL|        219|          206|
|59828450|  NULL|        209|          208|
|66087132|  NULL|        185|          184|
+--------+------+-----------+-------------+



1062656

## Verificación

In [3]:
# Totales (deben coincidir con la consulta 2 de Trino y de Dremio)
spark.sql(f"""
    SELECT COUNT(*) AS filas, SUM(total_votos) AS votos,
           SUM(votos_positivos) AS positivos, SUM(votos_negativos) AS negativos
    FROM {comun.TABLA_GOLD}
""").show()

# Registro puntual (consulta 4 de Trino y de Dremio)
spark.sql(f"SELECT * FROM {comun.TABLA_GOLD} WHERE post_id = 59837318").show(vertical=True, truncate=False)

+-------+-------+---------+---------+
|  filas|  votos|positivos|negativos|
+-------+-------+---------+---------+
|1062656|2013388|  1296738|   221594|
+-------+-------+---------+---------+



-RECORD 0----------------------------------------------------------------------------------------------------------------
 post_id         | 59837318                                                                                              
 post_type       | pregunta                                                                                              
 anio_post       | 2020                                                                                                  
 titulo          | Why is this "Hello, World!" JavaScript code fragment recognized as an acceptable program instruction? 
 tags            | javascript|node.js                                                                                    
 owner_user_id   | 7380417                                                                                               
 score_post      | 156                                                                                                   
 total_votos     | 172  

In [4]:
# Gold recalculado desde Silver vs Gold escrito: 0 y 0 = idénticos
from pyspark.sql import functions as F
recalculado = spark.sql(gold_agg.SQL_GOLD)
escrito = spark.table(comun.TABLA_GOLD).drop("fecha_cargue")
print("solo en recalculado:", recalculado.exceptAll(escrito.select(*recalculado.columns)).count())
print("solo en gold:       ", escrito.select(*recalculado.columns).exceptAll(recalculado).count())

solo en recalculado: 0


solo en gold:        0


In [5]:
spark.stop()